In [1]:
!pip install unsloth
!pip install trl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from datasets import load_dataset
from trl import SFTTrainer

In [3]:
from accelerate.utils.other import load
# get the model
max_seq_length = 2048
model_name = "unsloth/llama-3-8b-bnb-4bit" # Optimized 4-bit model

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.7: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:54: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [4]:
# Add lora adapters
model = FastLanguageModel.get_peft_model(
    model = model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

Unsloth 2026.3.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
dataset = load_dataset("json", data_files = "akbar_birbal_data.jsonl", split = "train")
dataset[:1]

Generating train split: 0 examples [00:00, ? examples/s]

{'instruction': ['Akbar asks a trick question.'],
 'input': ['Birbal, tell me, how many crows are there in our kingdom?'],
 'output': ["Birbal bowed and smiled. 'Jahapanah there are ninety-five thousand, four hundred and sixty-three, Majesty. If there are more, their relatives are visiting; if fewer, they have gone to visit theirs.'"]}

The `unsloth/llama-3-8b-bnb-4bit` model, like most large language models, expects tokenized input. When fine-tuning with `SFTTrainer`, it's common to format your data into a conversational turn. The Llama-3 models typically use a specific chat template for this. Let's define a simple prompt template based on your `instruction`, `input`, and `output` fields from the dataset.

In [8]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # If there's an input, combine instruction and input
        if input: # If there's an additional input, combine it with the instruction
            text = f"### Instruction:\n{instruction}\n### Input:\n{input}\n### Response:\n{output}"
        else: # Otherwise, just use the instruction
            text = f"### Instruction:\n{instruction}\n### Response:\n{output}"
        texts.append(text)
    return { "text" : texts, }

alpha_data = dataset.map(formatting_prompts_func, batched = True)
print(alpha_data[:1])

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

{'instruction': ['Akbar asks a trick question.'], 'input': ['Birbal, tell me, how many crows are there in our kingdom?'], 'output': ["Birbal bowed and smiled. 'Jahapanah there are ninety-five thousand, four hundred and sixty-three, Majesty. If there are more, their relatives are visiting; if fewer, they have gone to visit theirs.'"], 'text': ["### Instruction:\nAkbar asks a trick question.\n### Input:\nBirbal, tell me, how many crows are there in our kingdom?\n### Response:\nBirbal bowed and smiled. 'Jahapanah there are ninety-five thousand, four hundred and sixty-three, Majesty. If there are more, their relatives are visiting; if fewer, they have gone to visit theirs.'"]}


In [12]:
# Now set the trainer
trainer = SFTTrainer(
    model = model,
    train_dataset = alpha_data,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "akbar-birbal-model",
    ),
)

In [13]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,0.279390
2,0.279628
3,0.265159
4,0.198104
5,0.175014
6,0.118599
7,0.105290
8,0.076742
9,0.074514
10,0.060456


TrainOutput(global_step=60, training_loss=0.043599108203003806, metrics={'train_runtime': 243.004, 'train_samples_per_second': 1.975, 'train_steps_per_second': 0.247, 'total_flos': 2385187540377600.0, 'train_loss': 0.043599108203003806, 'epoch': 60.0})

In [19]:
# lets test the model
# get the input to the model
model_input_string = """{  'instruction': 'Akbar asks a trick question.',   'input': 'Birbal, if I am the sun, then what are you?' }"""


In [20]:
import json

def format_inference_prompt(instruction, input_text):
    if input_text:
        return f"### Instruction:\n{instruction}\n### Input:\n{input_text}\n### Response:\n"
    else:
        return f"### Instruction:\n{instruction}\n### Response:\n"

# Parse the input string
input_dict = json.loads(model_input_string.replace("'", '"'))

# Prepare the prompt for inference
inference_prompt = format_inference_prompt(input_dict['instruction'], input_dict['input'])

print("Inference Prompt:")
print(inference_prompt)

Inference Prompt:
### Instruction:
Akbar asks a trick question.
### Input:
Birbal, if I am the sun, then what are you?
### Response:



In [21]:
inputs = tokenizer(inference_prompt, return_tensors = "pt")

In [22]:
inputs

{'input_ids': tensor([[128000,  14711,  30151,    512,  56902,   2308,  17501,    264,  14397,
           3488,    627,  14711,   5688,    512, 105812,  13616,     11,    422,
            358,   1097,    279,   7160,     11,   1243,   1148,    527,    499,
           5380,  14711,   6075,    512]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])}

In [27]:
output = model.generate(
    input_ids = inputs["input_ids"].to(model.device),
    attention_mask = inputs["attention_mask"].to(model.device),
    max_new_tokens = 128,
    use_cache = False, # Changed to False to prevent the shape mismatch error
    temperature = 0.7,
    min_p = 0.1,
)
response = tokenizer.decode(output[0], skip_special_tokens = True)
print(response)

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

### Instruction:
Akbar asks a trick question.
### Input:
Birbal, if I am the sun, then what are you?
### Response:
Birbal bowed and replied, 'You are the shining light that removes the darkness from our world, Majesty. I am the reflected light that shows the way where you are not present.'
